# Phase 5 Benchmark: TPC-H Q3 (Shipping Priority)

Three-way inner join + filter + groupby + aggregation + sort + limit.

**Query:** For a given market segment ("BUILDING"), find the top-10 unshipped orders
by revenue. Revenue = `l_extendedprice * (1 - l_discount)`.

```sql
SELECT l_orderkey, o_orderdate, o_shippriority,
       SUM(l_extendedprice * (1 - l_discount)) AS revenue
FROM   customer JOIN orders    ON c_custkey = o_custkey
                JOIN lineitem  ON l_orderkey = o_orderkey
WHERE  c_mktsegment = 'BUILDING'
  AND  o_orderdate  < DATE '1995-03-15'   -- encoded as int
  AND  l_shipdate   > DATE '1995-03-15'   -- encoded as int
GROUP BY l_orderkey, o_orderdate, o_shippriority
ORDER BY revenue DESC, o_orderdate ASC
LIMIT 10;
```

In [ ]:
#| hide
#| eval: false

import time
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
from mxframe.lazy_expr import col, lit
from mxframe.lazy_frame import LazyFrame


## 1) Synthetic TPC-H Data (customer / orders / lineitem)

In [ ]:
#| hide
#| eval: false


DATE_1995_03_15 = 9204  # days since 1970-01-01

def make_tpch_tables(n_customers=15_000, n_orders=150_000, n_lineitem=600_000, seed=42):
    rng = np.random.default_rng(seed)

    # -- Customer --
    segments = np.array(["BUILDING", "AUTOMOBILE", "MACHINERY", "HOUSEHOLD", "FURNITURE"])
    c_custkey = np.arange(1, n_customers + 1, dtype=np.int32)
    c_mktsegment = rng.choice(segments, size=n_customers)
    customer = pa.table({
        "c_custkey": c_custkey,
        "c_mktsegment": c_mktsegment.tolist(),
    })

    # -- Orders --
    o_orderkey = np.arange(1, n_orders + 1, dtype=np.int32)
    o_custkey = rng.integers(1, n_customers + 1, size=n_orders, dtype=np.int32)
    o_orderdate = rng.integers(8800, 9300, size=n_orders, dtype=np.int32)  # ~1994-1995
    o_shippriority = rng.integers(0, 5, size=n_orders, dtype=np.int32)
    orders = pa.table({
        "o_orderkey": o_orderkey,
        "o_custkey": o_custkey,
        "o_orderdate": o_orderdate,
        "o_shippriority": o_shippriority,
    })

    # -- Lineitem --
    l_orderkey = rng.integers(1, n_orders + 1, size=n_lineitem, dtype=np.int32)
    l_extendedprice = rng.uniform(900.0, 100_000.0, size=n_lineitem).astype(np.float32)
    l_discount = rng.uniform(0.0, 0.10, size=n_lineitem).astype(np.float32)
    l_shipdate = rng.integers(8900, 9400, size=n_lineitem, dtype=np.int32)  # ~1994-1995
    lineitem = pa.table({
        "l_orderkey": l_orderkey,
        "l_extendedprice": l_extendedprice,
        "l_discount": l_discount,
        "l_shipdate": l_shipdate,
    })

    return customer, orders, lineitem

customer, orders, lineitem = make_tpch_tables()
print(f"customer:  {customer.num_rows:>10,} rows  cols={customer.column_names}")
print(f"orders:    {orders.num_rows:>10,} rows  cols={orders.column_names}")
print(f"lineitem:  {lineitem.num_rows:>10,} rows  cols={lineitem.column_names}")


customer:      15,000 rows  cols=['c_custkey', 'c_mktsegment']
orders:       150,000 rows  cols=['o_orderkey', 'o_custkey', 'o_orderdate', 'o_shippriority']
lineitem:     600,000 rows  cols=['l_orderkey', 'l_extendedprice', 'l_discount', 'l_shipdate']


## 2) Query Implementations

In [ ]:
#| hide
#| eval: false

# -- MXFrame Q3 (CPU) --
def run_q3_mxframe(customer, orders, lineitem, device="cpu"):
    lf_c = LazyFrame(customer).filter(col("c_mktsegment") == lit("BUILDING"))
    lf_o = LazyFrame(orders).filter(col("o_orderdate") < lit(DATE_1995_03_15))
    lf_l = LazyFrame(lineitem).filter(col("l_shipdate") > lit(DATE_1995_03_15))

    result = (
        lf_o
        .join(lf_c, left_on="o_custkey", right_on="c_custkey")
        .join(lf_l, left_on="o_orderkey", right_on="l_orderkey")
    )
    return result.compute(device=device)


# -- Pandas Q3 --
def run_q3_pandas(customer, orders, lineitem):
    import pandas as pd
    c = customer.to_pandas()
    o = orders.to_pandas()
    l = lineitem.to_pandas()

    c = c[c["c_mktsegment"] == "BUILDING"]
    o = o[o["o_orderdate"] < DATE_1995_03_15]
    l = l[l["l_shipdate"] > DATE_1995_03_15]

    merged = o.merge(c, left_on="o_custkey", right_on="c_custkey")
    merged = merged.merge(l, left_on="o_orderkey", right_on="l_orderkey")
    merged["revenue"] = merged["l_extendedprice"] * (1 - merged["l_discount"])
    result = (
        merged.groupby(["o_orderkey", "o_orderdate", "o_shippriority"])["revenue"]
        .sum()
        .reset_index()
        .sort_values(["revenue", "o_orderdate"], ascending=[False, True])
        .head(10)
    )
    return result


# -- Polars Q3 --
def run_q3_polars(customer, orders, lineitem):
    import polars as pl
    c = pl.from_arrow(customer).filter(pl.col("c_mktsegment") == "BUILDING")
    o = pl.from_arrow(orders).filter(pl.col("o_orderdate") < DATE_1995_03_15)
    l = pl.from_arrow(lineitem).filter(pl.col("l_shipdate") > DATE_1995_03_15)

    result = (
        o.join(c, left_on="o_custkey", right_on="c_custkey")
        .join(l, left_on="o_orderkey", right_on="l_orderkey")
        .with_columns((pl.col("l_extendedprice") * (1 - pl.col("l_discount"))).alias("revenue"))
        .group_by(["o_orderkey", "o_orderdate", "o_shippriority"])
        .agg(pl.col("revenue").sum())
        .sort(["revenue", "o_orderdate"], descending=[True, False])
        .head(10)
    )
    return result.to_arrow()


# -- DuckDB Q3 (correctness reference) --
def run_q3_duckdb(customer, orders, lineitem):
    import duckdb
    con = duckdb.connect()
    con.execute("CREATE TABLE customer AS SELECT * FROM customer")
    con.execute("CREATE TABLE orders AS SELECT * FROM orders")
    con.execute("CREATE TABLE lineitem AS SELECT * FROM lineitem")
    result = con.execute(f"""
        SELECT l_orderkey, o_orderdate, o_shippriority,
               SUM(l_extendedprice * (1 - l_discount)) AS revenue
        FROM   customer
        JOIN   orders   ON c_custkey = o_custkey
        JOIN   lineitem ON l_orderkey = o_orderkey
        WHERE  c_mktsegment = 'BUILDING'
          AND  o_orderdate  < {DATE_1995_03_15}
          AND  l_shipdate   > {DATE_1995_03_15}
        GROUP BY l_orderkey, o_orderdate, o_shippriority
        ORDER BY revenue DESC, o_orderdate ASC
        LIMIT 10
    """).arrow()
    return result


## 3) Correctness Preview

In [ ]:
#| hide
#| eval: false


# Quick MXFrame result preview
mx_result = run_q3_mxframe(customer, orders, lineitem)
print(f"MXFrame join result: {mx_result.num_rows:,} rows, cols={mx_result.column_names}")
print(mx_result.to_pandas().head(10))


MXFrame join result: 38,606 rows, cols=['o_orderkey', 'o_custkey', 'o_orderdate', 'o_shippriority', 'c_mktsegment', 'l_extendedprice', 'l_discount', 'l_shipdate']
   o_orderkey  o_custkey  o_orderdate  o_shippriority c_mktsegment  \
0          12      11034         8936               1     BUILDING   
1          24       8178         8978               3     BUILDING   
2          27      10066         9074               3     BUILDING   
3          30      10746         9005               1     BUILDING   
4          42        888         9008               3     BUILDING   
5          42        888         9008               3     BUILDING   
6          42        888         9008               3     BUILDING   
7          43       8117         9194               0     BUILDING   
8          43       8117         9194               0     BUILDING   
9          48       2504         8840               4     BUILDING   

   l_extendedprice  l_discount  l_shipdate  
0     94291.789062   

: 

## 4) Benchmark

In [ ]:
#| hide
#| eval: false


def bench(fn, *args, warmup=1, runs=5):
    for _ in range(warmup):
        fn(*args)
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn(*args)
        times.append(time.perf_counter() - t0)
    return np.median(times) * 1000  # ms

print("Benchmarking TPC-H Q3 (median of 5 runs, 1 warmup)...")
print(f"  customer={customer.num_rows:,}  orders={orders.num_rows:,}  lineitem={lineitem.num_rows:,}")
print()

t_mx = bench(run_q3_mxframe, customer, orders, lineitem)
print(f"  MXFrame CPU (Mojo join)  :  {t_mx:8.1f} ms")

try:
    t_mx_gpu = bench(run_q3_mxframe, customer, orders, lineitem, "gpu")
    print(f"  MXFrame GPU (Mojo join)  :  {t_mx_gpu:8.1f} ms  ({t_mx/t_mx_gpu:.1f}x vs CPU)")
except Exception as e:
    print(f"  MXFrame GPU (Mojo join)  :  FAILED — {e}")

try:
    t_pd = bench(run_q3_pandas, customer, orders, lineitem)
    print(f"  Pandas                   :  {t_pd:8.1f} ms  ({t_pd/t_mx:.1f}x vs MXFrame CPU)")
except Exception as e:
    print(f"  Pandas                   :  FAILED — {e}")

try:
    t_pl = bench(run_q3_polars, customer, orders, lineitem)
    print(f"  Polars                   :  {t_pl:8.1f} ms  ({t_pl/t_mx:.1f}x vs MXFrame CPU)")
except Exception as e:
    print(f"  Polars                   :  FAILED — {e}")

try:
    t_dk = bench(run_q3_duckdb, customer, orders, lineitem)
    print(f"  DuckDB                   :  {t_dk:8.1f} ms  ({t_dk/t_mx:.1f}x vs MXFrame CPU)")
except Exception as e:
    print(f"  DuckDB                   :  FAILED — {e}")


Benchmarking TPC-H Q3 (median of 5 runs, 1 warmup)...
  customer=15,000  orders=150,000  lineitem=600,000

  MXFrame CPU (Mojo join)  :      14.6 ms
